# Lab 4 v2 — HOG + SVM Face Detector (improved)

**Fixes vs v1:**
- Negatives come from real background images (skimage), not face transforms — the original flipped/rotated faces are still face-like, so the model never learns what non-faces look like
- Horizontal flip augmentation on positives doubles training set
- Hard negative mining: after initial training, run detector on background images, collect false positives, retrain
- Gender SVM uses `class_weight='balanced'` — LFW skews ~80% male
- Architecture unchanged: same HOG, same LinearSVC, same sliding window + pyramid

In [ ]:
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import pickle
import os
from pathlib import Path

from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.datasets import fetch_lfw_people
import skimage.data as skd

%matplotlib inline

In [ ]:
print('Loading LFW...')
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
WIN_H, WIN_W = lfw.images.shape[1], lfw.images.shape[2]
print(f'Images: {lfw.images.shape[0]}, window: {WIN_H}x{WIN_W}, persons: {len(lfw.target_names)}')
print('Names:', lfw.target_names)

In [ ]:
GENDER_LABELS = {
    'Ariel Sharon': 0, 'Colin Powell': 0, 'Donald Rumsfeld': 0,
    'George W Bush': 0, 'Gerhard Schroeder': 0, 'Hugo Chavez': 0,
    'Tony Blair': 0, 'Junichiro Koizumi': 0, 'Jean Chretien': 0,
    'John Ashcroft': 0, 'Vladimir Putin': 0, 'Vladmir Putin': 0, 'Hamid Karzai': 0,
    'Luiz Inacio Lula da Silva': 0, 'Jacques Chirac': 0, 'Jiang Zemin': 0,
    'Vicente Fox': 0, 'Silvio Berlusconi': 0, 'Alejandro Toledo': 0,
    'John Snow': 0, 'Arnold Schwarzenegger': 0,
    'Lleyton Hewitt': 0, 'Andre Agassi': 0, 'Pete Sampras': 0,
    'Roger Federer': 0, 'Tiger Woods': 0, 'Lance Armstrong': 0,
    'David Beckham': 0, 'Kofi Annan': 0, 'Fidel Castro': 0,
    'Recep Tayyip Erdogan': 0, 'Pervez Musharraf': 0, 'Carlos Menem': 0,
    'Gray Davis': 0, 'Bill Clinton': 0, 'Spencer Abraham': 0,
    'Tom Daschle': 0, 'Tom Ridge': 0, 'Paul Bremer': 0,
    'Nestor Kirchner': 0, 'Atal Bihari Vajpayee': 0, 'Nabil Shaath': 0,
    'Mahmoud Abbas': 0, 'Alvaro Uribe': 0, 'Eduardo Duhalde': 0,
    'Roh Moo-hyun': 0, 'Ariel Sharon': 0, 'Jose Maria Aznar': 0,
    'Juan Carlos Ferrero': 0, 'Joschka Fischer': 0,
    # female
    'Jennifer Aniston': 1, 'Jennifer Capriati': 1, 'Jennifer Lopez': 1,
    'Julianne Moore': 1, 'Halle Berry': 1, 'Winona Ryder': 1,
    'Condoleezza Rice': 1, 'Serena Williams': 1, 'Venus Williams': 1,
    'Naomi Watts': 1, 'Gloria Macapagal Arroyo': 1, 'Megawati Sukarnoputri': 1,
    'Renee Zellweger': 1, 'Meryl Streep': 1, 'Sharon Stone': 1,
    'Zhu Rongji': 0,
}

## HOG descriptor
Architecture unchanged: central differences → cell histograms → L2-normalized blocks.

In [ ]:
def to_gray(image_float):
    if len(image_float.shape) == 3:
        return (0.299*image_float[:,:,0] +
                0.587*image_float[:,:,1] +
                0.114*image_float[:,:,2]) * 255.0
    return image_float * 255.0

def compute_gradients(gray):
    Ix = np.zeros_like(gray)
    Iy = np.zeros_like(gray)
    Ix[:, 1:-1] = (gray[:, 2:] - gray[:, :-2]) / 2.0
    Iy[1:-1, :] = (gray[2:, :] - gray[:-2, :]) / 2.0
    magnitude = np.sqrt(Ix**2 + Iy**2)
    angle     = np.degrees(np.arctan2(Iy, Ix)) % 180.0
    return magnitude, angle

def build_cell_histograms(magnitude, angle, cell_size=8, num_bins=9):
    h, w   = magnitude.shape
    n_cy   = h // cell_size
    n_cx   = w // cell_size
    hists  = np.zeros((n_cy, n_cx, num_bins))
    for cy in range(n_cy):
        for cx in range(n_cx):
            r0, r1 = cy*cell_size, (cy+1)*cell_size
            c0, c1 = cx*cell_size, (cx+1)*cell_size
            hist, _ = np.histogram(angle[r0:r1, c0:c1], bins=num_bins,
                                   range=(0, 180), weights=magnitude[r0:r1, c0:c1])
            hists[cy, cx] = hist
    return hists

def normalize_blocks(cell_hists, block_size=2):
    n_cy, n_cx, num_bins = cell_hists.shape
    descriptor = []
    for by in range(n_cy - block_size + 1):
        for bx in range(n_cx - block_size + 1):
            block = cell_hists[by:by+block_size, bx:bx+block_size].flatten()
            norm  = np.sqrt(np.sum(block**2) + 1e-6)
            descriptor.append(block / norm)
    return np.concatenate(descriptor)

def hog_descriptor(image_float, cell_size=8, num_bins=9, block_size=2):
    gray        = to_gray(image_float)
    mag, ang    = compute_gradients(gray)
    hists       = build_cell_histograms(mag, ang, cell_size, num_bins)
    return normalize_blocks(hists, block_size)

# sanity check
sample_desc = hog_descriptor(lfw.images[0])
print('HOG feature dim:', sample_desc.shape[0])

## Face features with flip augmentation

In [ ]:
np.random.seed(42)

X_faces  = []
y_gender = []
valid_mask = []

for i, (img, target) in enumerate(zip(lfw.images, lfw.target)):
    name = lfw.target_names[target]
    if name not in GENDER_LABELS:
        continue
    g = GENDER_LABELS[name]
    X_faces.append(hog_descriptor(img))
    y_gender.append(g)
    valid_mask.append(i)
    # horizontal flip augmentation — face is symmetric, flip is still a face
    X_faces.append(hog_descriptor(img[:, ::-1, :]))
    y_gender.append(g)
    valid_mask.append(i)

X_faces  = np.array(X_faces)
y_gender = np.array(y_gender)

n_m = np.sum(y_gender == 0)
n_f = np.sum(y_gender == 1)
print(f'Face features: {X_faces.shape}  |  Male: {n_m}  Female: {n_f}  Ratio: {n_m/max(n_f,1):.1f}x')

## Background negatives

The original approach used flipped/rotated faces as negatives — those still look like faces to HOG, so the detector never learned what *non*-faces look like. Here we use crops from real background images (skimage built-ins: nature, textures, objects) — genuine non-face content.

In [ ]:
def get_background_images():
    loaders = [
        skd.astronaut, skd.coffee, skd.chelsea, skd.rocket,
        skd.immunohistochemistry, skd.grass, skd.brick, skd.gravel,
        skd.coins, skd.camera, skd.clock, skd.page,
    ]
    imgs = []
    for loader in loaders:
        try:
            img = loader()
            if img.ndim == 2:
                img = np.stack([img]*3, axis=-1)
            elif img.shape[2] == 4:
                img = img[:, :, :3]
            imgs.append(img.astype(np.float32) / 255.0)
        except Exception as e:
            print(f'  skip {loader.__name__}: {e}')
    return imgs

def random_crops(img_float, win_h, win_w, n):
    h, w = img_float.shape[:2]
    crops = []
    attempts = 0
    while len(crops) < n and attempts < n * 5:
        attempts += 1
        if h <= win_h or w <= win_w:
            break
        r = np.random.randint(0, h - win_h)
        c = np.random.randint(0, w - win_w)
        crops.append(img_float[r:r+win_h, c:c+win_w])
    return crops

bg_images = get_background_images()
print(f'Background images loaded: {len(bg_images)}')

n_faces = len(X_faces)
crops_per_img = n_faces // len(bg_images) + 20

bg_crops = []
for bg in bg_images:
    bg_crops.extend(random_crops(bg, WIN_H, WIN_W, crops_per_img))

# pure random noise
noise_patches = [np.random.rand(WIN_H, WIN_W, 3).astype(np.float32)
                 for _ in range(n_faces // 4)]

all_easy_neg = bg_crops + noise_patches
print(f'Easy negatives before HOG: {len(all_easy_neg)}')

print('Computing HOG for easy negatives...')
X_neg_easy = np.array([hog_descriptor(p) for p in all_easy_neg])
print(f'Easy negatives matrix: {X_neg_easy.shape}')

## Initial detector (easy negatives only)

In [ ]:
# balance: equal pos/neg
if len(X_neg_easy) > n_faces:
    idx = np.random.choice(len(X_neg_easy), n_faces, replace=False)
    X_neg_init = X_neg_easy[idx]
else:
    X_neg_init = X_neg_easy

X_init = np.vstack([X_faces, X_neg_init])
y_init = np.concatenate([np.ones(len(X_faces)), -np.ones(len(X_neg_init))])

X_tr0, X_te0, y_tr0, y_te0 = train_test_split(
    X_init, y_init, test_size=0.2, random_state=42, stratify=y_init
)

det_init = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    LinearSVC(C=0.01, max_iter=5000))
])
det_init.fit(X_tr0, y_tr0)
print(f'Initial detector accuracy: {accuracy_score(y_te0, det_init.predict(X_te0)):.3f}')

## Hard negative mining

Run the initial detector over background images with a dense sliding window. Any patch that the detector *misclassifies as a face* is a hard negative — it's the most informative training signal to push false positives down.

In [ ]:
def mine_hard_negatives(detector, images_float, win_h, win_w,
                         step=8, threshold=0.0, max_total=3000):
    hard = []
    for bg in images_float:
        bg_u8 = (bg * 255).astype(np.uint8)
        h, w  = bg_u8.shape[:2]
        for r in range(0, h - win_h + 1, step):
            for c in range(0, w - win_w + 1, step):
                patch   = bg_u8[r:r+win_h, c:c+win_w]
                patch_f = patch.astype(np.float32) / 255.0
                desc    = hog_descriptor(patch_f).reshape(1, -1)
                score   = detector.decision_function(desc)[0]
                if score > threshold:
                    hard.append((score, hog_descriptor(patch_f)))
    if not hard:
        print('  No hard negatives found — initial detector is already clean.')
        return np.array([]).reshape(0, win_h)  # placeholder
    # keep hardest ones
    hard.sort(key=lambda x: x[0], reverse=True)
    hard = hard[:max_total]
    return np.array([h[1] for h in hard])

print('Mining hard negatives (step=8 on all bg images)...')
X_hard = mine_hard_negatives(det_init, bg_images, WIN_H, WIN_W,
                              step=8, threshold=0.0, max_total=3000)
print(f'Hard negatives found: {len(X_hard)}')

## Final detector: retrain with hard negatives

In [ ]:
if len(X_hard) > 0 and X_hard.ndim == 2 and X_hard.shape[1] == X_faces.shape[1]:
    X_neg_all = np.vstack([X_neg_easy, X_hard])
else:
    X_neg_all = X_neg_easy

# allow up to 2x negatives to reduce false positive rate
max_neg = 2 * len(X_faces)
if len(X_neg_all) > max_neg:
    idx = np.random.choice(len(X_neg_all), max_neg, replace=False)
    X_neg_all = X_neg_all[idx]

X_det = np.vstack([X_faces, X_neg_all])
y_det = np.concatenate([np.ones(len(X_faces)), -np.ones(len(X_neg_all))])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_det, y_det, test_size=0.2, random_state=42, stratify=y_det
)

detector_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    LinearSVC(C=0.1, max_iter=5000))
])
detector_pipe.fit(X_tr, y_tr)

y_pred = detector_pipe.predict(X_te)
print(f'Final detector accuracy: {accuracy_score(y_te, y_pred):.3f}')
print(classification_report(y_te, y_pred, target_names=['not-face', 'face']))

## Gender classifier

In [ ]:
X_tr_g, X_te_g, y_tr_g, y_te_g = train_test_split(
    X_faces, y_gender, test_size=0.2, random_state=42, stratify=y_gender
)

gender_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    LinearSVC(C=1.0, class_weight='balanced', max_iter=5000))
])
gender_pipe.fit(X_tr_g, y_tr_g)

y_pred_g = gender_pipe.predict(X_te_g)
print(f'Gender accuracy: {accuracy_score(y_te_g, y_pred_g):.3f}')
print(classification_report(y_te_g, y_pred_g, target_names=['male', 'female']))

## Confusion matrices

In [ ]:
cm_det = confusion_matrix(y_te,   detector_pipe.predict(X_te),   labels=[-1, 1])
cm_gen = confusion_matrix(y_te_g, gender_pipe.predict(X_te_g),   labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, cm, title, labels in [
    (axes[0], cm_det, 'Detector: face / not-face', ['not-face', 'face']),
    (axes[1], cm_gen, 'Gender: male / female',     ['male',     'female']),
]:
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
plt.tight_layout()
plt.show()

## Save models

In [ ]:
LABS_DIR = Path('.')

with open(LABS_DIR / 'detector_model.pkl', 'wb') as f:
    pickle.dump(detector_pipe, f)
with open(LABS_DIR / 'gender_model.pkl', 'wb') as f:
    pickle.dump(gender_pipe, f)

print('Saved: detector_model.pkl  gender_model.pkl')

## Inference — 10 configs × all images in `totest/`

Each config produces one PNG in `results/config_NN.png`. A combined grid is saved to `batch_results.png`.

In [ ]:
CONFIGS = [
    {"name": "thresh=0.4 step=8  scale=0.90 iou=0.30", "threshold": 0.4, "step": 8,  "pyramid_scale": 0.90, "iou_thresh": 0.30},
    {"name": "thresh=0.6 step=8  scale=0.90 iou=0.30", "threshold": 0.6, "step": 8,  "pyramid_scale": 0.90, "iou_thresh": 0.30},
    {"name": "thresh=0.8 step=16 scale=0.85 iou=0.30", "threshold": 0.8, "step": 16, "pyramid_scale": 0.85, "iou_thresh": 0.30},
    {"name": "thresh=1.0 step=16 scale=0.85 iou=0.30", "threshold": 1.0, "step": 16, "pyramid_scale": 0.85, "iou_thresh": 0.30},
    {"name": "thresh=1.2 step=16 scale=0.85 iou=0.35", "threshold": 1.2, "step": 16, "pyramid_scale": 0.85, "iou_thresh": 0.35},
    {"name": "thresh=1.5 step=16 scale=0.85 iou=0.35", "threshold": 1.5, "step": 16, "pyramid_scale": 0.85, "iou_thresh": 0.35},
    {"name": "thresh=1.8 step=24 scale=0.80 iou=0.35", "threshold": 1.8, "step": 24, "pyramid_scale": 0.80, "iou_thresh": 0.35},
    {"name": "thresh=1.0 step=24 scale=0.80 iou=0.25", "threshold": 1.0, "step": 24, "pyramid_scale": 0.80, "iou_thresh": 0.25},
    {"name": "thresh=0.8 step=8  scale=0.75 iou=0.20", "threshold": 0.8, "step": 8,  "pyramid_scale": 0.75, "iou_thresh": 0.20},
    {"name": "thresh=2.0 step=16 scale=0.85 iou=0.40", "threshold": 2.0, "step": 16, "pyramid_scale": 0.85, "iou_thresh": 0.40},
]

SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

In [ ]:
def image_pyramid(image_uint8, scale=0.85, min_size=64):
    img    = image_uint8.copy()
    factor = 1.0
    while True:
        yield img, factor
        h, w   = img.shape[:2]
        new_h  = int(h * scale)
        new_w  = int(w * scale)
        if new_h < min_size or new_w < min_size:
            break
        img    = cv2.resize(img, (new_w, new_h))
        factor *= scale

def iou(boxA, boxB):
    r0 = max(boxA[0], boxB[0]); c0 = max(boxA[1], boxB[1])
    r1 = min(boxA[2], boxB[2]); c1 = min(boxA[3], boxB[3])
    inter = max(0, r1 - r0) * max(0, c1 - c0)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def nms(detections, iou_thresh=0.3):
    if not detections:
        return []
    detections = sorted(detections, key=lambda x: x[0], reverse=True)
    kept = []
    while detections:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections if iou(best[1:], d[1:]) < iou_thresh]
    return kept

def detect_and_classify(image_rgb, detector, gender_clf, cfg):
    detections = []
    for img_scaled, factor in image_pyramid(image_rgb, scale=cfg['pyramid_scale']):
        h_s, w_s = img_scaled.shape[:2]
        if h_s < WIN_H or w_s < WIN_W:
            continue
        for r in range(0, h_s - WIN_H + 1, cfg['step']):
            for c in range(0, w_s - WIN_W + 1, cfg['step']):
                patch   = img_scaled[r:r+WIN_H, c:c+WIN_W]
                patch_f = patch.astype(np.float32) / 255.0
                desc    = hog_descriptor(patch_f).reshape(1, -1)
                score   = detector.decision_function(desc)[0]
                if score > cfg['threshold']:
                    r0 = int(r / factor); c0 = int(c / factor)
                    r1 = int((r + WIN_H) / factor); c1 = int((c + WIN_W) / factor)
                    detections.append((score, r0, c0, r1, c1))
    detections = nms(detections, cfg['iou_thresh'])
    results = []
    for score, r0, c0, r1, c1 in detections:
        crop = image_rgb[r0:r1, c0:c1]
        if crop.size == 0:
            continue
        face_r = cv2.resize(crop, (WIN_W, WIN_H))
        face_f = face_r.astype(np.float32) / 255.0
        desc   = hog_descriptor(face_f).reshape(1, -1)
        gender = int(gender_clf.predict(desc)[0])
        results.append((r0, c0, r1, c1, gender, float(score)))
    return results

In [ ]:
def draw_on_ax(ax, image_rgb, results, title):
    ax.imshow(image_rgb)
    ax.axis('off')
    ax.set_title(title, fontsize=6, pad=3, color='white',
                 bbox=dict(facecolor='#1a1a2e', edgecolor='none', pad=2))
    for (r0, c0, r1, c1, gender, score) in results:
        color = '#4fc3f7' if gender == 0 else '#f48fb1'
        label = f"{'M' if gender == 0 else 'F'} {score:.2f}"
        rect  = mpatches.Rectangle((c0, r0), c1-c0, r1-r0,
                                    linewidth=1.2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(c0+2, r0-3, label, fontsize=5, color=color,
                bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))
    n_m = sum(1 for *_, g, _s in results if g == 0)
    n_f = sum(1 for *_, g, _s in results if g == 1)
    ax.text(0.02, 0.98, f'Faces:{len(results)} M:{n_m} F:{n_f}',
            transform=ax.transAxes, fontsize=5.5, color='yellow', va='top',
            bbox=dict(facecolor='black', alpha=0.6, pad=1.5, edgecolor='none'))


def save_config_png(images_rgb, img_names, results_col, cfg, out_path):
    n = len(images_rgb)
    CELL_W, CELL_H, LABEL_W = 3.2, 2.6, 1.0
    fig = plt.figure(figsize=(LABEL_W + CELL_W, n * CELL_H + 0.5), facecolor='#0d0d1a')
    gs  = GridSpec(n, 2, figure=fig,
                   width_ratios=[LABEL_W / CELL_W, 1.0],
                   wspace=0.03, hspace=0.15,
                   left=0.01, right=0.99, top=0.94, bottom=0.02)
    fig.suptitle(cfg['name'], color='white', fontsize=8, y=0.98, va='top', fontfamily='monospace')
    for i in range(n):
        ax_l = fig.add_subplot(gs[i, 0])
        ax_l.set_facecolor('#0d0d1a'); ax_l.axis('off')
        ax_l.text(0.95, 0.5, img_names[i], transform=ax_l.transAxes,
                  fontsize=6, color='#a0c4ff', va='center', ha='right', fontfamily='monospace')
        ax = fig.add_subplot(gs[i, 1])
        ax.set_facecolor('#0d0d1a')
        draw_on_ax(ax, images_rgb[i], results_col[i], cfg['name'])
    fig.savefig(str(out_path), dpi=120, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close(fig)


def save_combined_png(images_rgb, img_names, results_grid, out_path):
    n_imgs = len(images_rgb)
    n_cfgs = len(CONFIGS)
    CELL_W, CELL_H, LABEL_W = 3.2, 2.6, 1.0
    fig = plt.figure(figsize=(LABEL_W + n_cfgs * CELL_W, n_imgs * CELL_H + 0.6), facecolor='#0d0d1a')
    gs  = GridSpec(n_imgs, 1 + n_cfgs, figure=fig,
                   width_ratios=[LABEL_W / CELL_W] + [1.0] * n_cfgs,
                   wspace=0.04, hspace=0.15,
                   left=0.01, right=0.99, top=0.97, bottom=0.01)
    fig.suptitle(
        f'HOG+SVM v2 — {n_imgs} images × {n_cfgs} configs  |  Blue=Male  Pink=Female',
        color='white', fontsize=10, y=0.99, va='top', fontfamily='monospace'
    )
    for i in range(n_imgs):
        ax_l = fig.add_subplot(gs[i, 0])
        ax_l.set_facecolor('#0d0d1a'); ax_l.axis('off')
        ax_l.text(0.95, 0.5, img_names[i], transform=ax_l.transAxes,
                  fontsize=6, color='#a0c4ff', va='center', ha='right', fontfamily='monospace')
        for j, cfg in enumerate(CONFIGS):
            ax = fig.add_subplot(gs[i, j + 1])
            ax.set_facecolor('#0d0d1a')
            draw_on_ax(ax, images_rgb[i], results_grid[i][j], cfg['name'])
    fig.savefig(str(out_path), dpi=120, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.close(fig)

In [ ]:
TOTEST_DIR = LABS_DIR / 'totest'
RESULTS_DIR = LABS_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_paths = sorted([p for p in TOTEST_DIR.iterdir()
                      if p.suffix.lower() in SUPPORTED_EXT])
print(f'Test images found: {len(image_paths)}')

images_rgb = []
img_names  = []
for p in image_paths:
    bgr = cv2.imread(str(p))
    if bgr is None:
        print(f'  skip (unreadable): {p.name}')
        continue
    h, w = bgr.shape[:2]
    if max(h, w) > 640:
        s   = 640 / max(h, w)
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)))
    images_rgb.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img_names.append(p.name)

print(f'Loaded: {len(images_rgb)} images')

In [ ]:
n_imgs = len(images_rgb)
n_cfgs = len(CONFIGS)
results_grid = [[None] * n_cfgs for _ in range(n_imgs)]

for j, cfg in enumerate(CONFIGS):
    print(f'[{j+1}/{n_cfgs}] {cfg["name"]}')
    for i, (img_rgb, name) in enumerate(zip(images_rgb, img_names)):
        try:
            res = detect_and_classify(img_rgb, detector_pipe, gender_pipe, cfg)
        except Exception as e:
            print(f'  WARN {name}: {e}')
            res = []
        results_grid[i][j] = res
        print(f'  {name}: {len(res)} face(s)  ' +
              ', '.join(f"({'M' if g==0 else 'F'} {s:.2f})" for *_, g, s in res))

    config_png = RESULTS_DIR / f'config_{j+1:02d}.png'
    save_config_png(images_rgb, img_names,
                    [results_grid[i][j] for i in range(n_imgs)],
                    cfg, config_png)
    print(f'  => {config_png.name}')

print('\nSaving combined grid...')
save_combined_png(images_rgb, img_names, results_grid, LABS_DIR / 'batch_results.png')
print('Done.')

In [ ]:
print('\n── Per-config summary ──')
for j, cfg in enumerate(CONFIGS):
    total = sum(len(results_grid[i][j]) for i in range(n_imgs))
    m     = sum(sum(1 for *_, g, _s in results_grid[i][j] if g == 0) for i in range(n_imgs))
    f     = sum(sum(1 for *_, g, _s in results_grid[i][j] if g == 1) for i in range(n_imgs))
    print(f'  [{j+1:2d}] {cfg["name"]:44s}  faces={total:3d}  M={m:3d}  F={f:3d}')